# Week 6: Listing Signal Extraction

This notebook extracts structured signals from real estate listings by combining the Week 3 entity extractor with taxonomy-based pattern matching.

In [67]:
import sys
sys.path.append("..")

from scripts.entity_extractor import EntityExtractor
from scripts.signal_extractor import SignalExtractor

In [68]:
entity_extractor = EntityExtractor(
    taxonomy_path="../data/processed/taxonomy.json"
)

signal_extractor = SignalExtractor(
    taxonomy_path="../data/processed/taxonomy.json",
    entity_extractor=entity_extractor
)

In [69]:
sample_listing = {
    "L_ListingID": "TEST001",
    "L_Remarks": """
    Beautiful renovated home with a pool and fireplace.
    Located near shopping and dining with easy freeway access.
    Seller financing available.
    """
}

signal_extractor.extract_signals(sample_listing)

{'listing_id': 'TEST001',
 'entities': {'bedrooms': None,
  'bathrooms': None,
  'price': None,
  'sqft': None,
  'amenities': ['fireplace', 'pool']},
 'amenities': ['fireplace', 'pool'],
 'condition_keywords': ['renovated'],
 'financing_terms': ['financing available', 'seller financing'],
 'location_features': ['freeway access', 'near shopping']}

In [70]:
{
    "listing_id": "TEST001",
    "entities": {
        "bedrooms": None,
        "bathrooms": None,
        "price": None,
        "sqft": None,
        "amenities": [...]
    },
    "amenities": ["fireplace", "pool"],
    "condition_keywords": ["renovated"],
    "financing_terms": ["seller financing"],
    "location_features": [
        "freeway access",
        "near shopping"
    ]
}

{'listing_id': 'TEST001',
 'entities': {'bedrooms': None,
  'bathrooms': None,
  'price': None,
  'sqft': None,
  'amenities': [Ellipsis]},
 'amenities': ['fireplace', 'pool'],
 'condition_keywords': ['renovated'],
 'financing_terms': ['seller financing'],
 'location_features': ['freeway access', 'near shopping']}

## Extraction Accuracy Evaluation

A sample of 50 listings is manually reviewed to evaluate extraction accuracy.

Structured fields include bedrooms, bathrooms, price, and square footage.

Free-text fields include amenities, condition keywords, financing terms, and location features.

In [71]:
import pandas as pd
import mysql.connector

signals_df = pd.read_json(
    "../data/processed/week6_listing_signals.jsonl",
    lines=True
)

evaluation_df = signals_df.sample(
    n=50,
    random_state=42
).copy()

len(evaluation_df)

50

In [72]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="real_estate"
)

cursor = conn.cursor()

listing_ids = evaluation_df["listing_id"].astype(str).tolist()

placeholders = ", ".join(["%s"] * len(listing_ids))

cursor.execute(
    f"""
    SELECT L_ListingID, L_Remarks
    FROM rets_property
    WHERE L_ListingID IN ({placeholders})
    """,
    listing_ids
)

remarks_df = pd.DataFrame(
    cursor.fetchall(),
    columns=["listing_id", "remarks"]
)

cursor.close()
conn.close()

remarks_df["listing_id"] = remarks_df["listing_id"].astype(str)
evaluation_df["listing_id"] = evaluation_df["listing_id"].astype(str)

evaluation_df = evaluation_df.merge(
    remarks_df,
    on="listing_id",
    how="left"
)

In [73]:
evaluation_df["bedrooms"] = evaluation_df["entities"].apply(
    lambda x: x.get("bedrooms")
)

evaluation_df["bathrooms"] = evaluation_df["entities"].apply(
    lambda x: x.get("bathrooms")
)

evaluation_df["price"] = evaluation_df["entities"].apply(
    lambda x: x.get("price")
)

evaluation_df["sqft"] = evaluation_df["entities"].apply(
    lambda x: x.get("sqft")
)

In [74]:
evaluation_sample = evaluation_df[
    [
        "listing_id",
        "remarks",
        "bedrooms",
        "bathrooms",
        "price",
        "sqft",
        "amenities",
        "condition_keywords",
        "financing_terms",
        "location_features"
    ]
].copy()

evaluation_sample["structured_correct"] = ""
evaluation_sample["free_text_correct"] = ""

In [75]:
evaluation_sample.to_csv(
    "../data/processed/week6_evaluation_sample_new.csv",
    index=False
)

In [76]:
evaluation_sample = pd.read_csv(
    "../data/processed/week6_evaluation_sample_labeled_final.csv"
)

structured_accuracy = evaluation_sample["structured_correct"].mean()
free_text_accuracy = evaluation_sample["free_text_correct"].mean()

print(f"Structured field accuracy: {structured_accuracy:.2%}")
print(f"Free-text accuracy: {free_text_accuracy:.2%}")

Structured field accuracy: 96.00%
Free-text accuracy: 86.00%
